# Recursive Peeling Algorithm Test

This notebook tests the recursive peeling algorithm on synthetic hierarchical data.

**Problem Setup:** Multiple linear regression with hierarchical coefficients:
- y = x₁ + 0.1x₂ + 0.01x₃ + 0.001x₄ + ... + noise
- Each coefficient is an order of magnitude smaller than the previous
- Tests the algorithm's ability to discover hierarchical structure

In [24]:
# Import Required Libraries
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add PySPIDER to path
sys.path.append('/Users/kevinzhou/Desktop/Projects/PySPIDER-1/src')

from PySPIDER.commons.sparse_reg_bf import (
    sparse_reg_bf, Scaler, Initializer, Residual, ModelIterator, Threshold
)
from PySPIDER.commons.sparse_reg_peeling import recursive_peeling_regression

print("Libraries imported successfully!")

Libraries imported successfully!


## Step 1: Generate Synthetic Hierarchical Data

Create synthetic data where the true coefficients follow a hierarchical pattern.

In [25]:
def generate_synthetic_data(n_samples=1000, n_features=10, noise_level=0.0, seed=42):
    """
    Generate synthetic data for hierarchical regression testing.
    
    Parameters:
    -----------
    n_samples : int
        Number of data samples
    n_features : int  
        Number of features (x₁, x₂, x₃, ...)
    noise_level : float
        Standard deviation of Gaussian noise to add
    seed : int
        Random seed for reproducibility
        
    Returns:
    --------
    X : np.ndarray, shape (n_samples, n_features)
        Feature matrix with normally distributed columns
    y : np.ndarray, shape (n_samples,)
        Target values from hierarchical linear model
    true_coefficients : np.ndarray, shape (n_features,)
        The true coefficients used to generate y
    """
    np.random.seed(seed)
    
    # Generate normally distributed features
    X = np.random.randn(n_samples, n_features)
    
    # Create hierarchical coefficients: 1, 0.1, 0.01, 0.001, ...
    true_coefficients = np.array([10**(-i) for i in range(n_features)])
    
    # Generate target values: y = X @ coefficients + noise
    y = X @ true_coefficients
    
    if noise_level > 0:
        noise = np.random.randn(n_samples) * noise_level
        y += noise
    
    return X, y, true_coefficients

# Generate test data with 10 features now
X, y, true_coeffs = generate_synthetic_data(n_samples=500, n_features=10, noise_level=0.0)

print(f"Data shape: X {X.shape}, y {y.shape}")
print(f"True coefficients: {true_coeffs}")
print(f"Target statistics: mean={y.mean():.4f}, std={y.std():.4f}")
print(f"Feature statistics: X mean={X.mean():.4f}, X std={X.std():.4f}")

Data shape: X (500, 10), y (500,)
True coefficients: [1.e+00 1.e-01 1.e-02 1.e-03 1.e-04 1.e-05 1.e-06 1.e-07 1.e-08 1.e-09]
Target statistics: mean=0.0052, std=0.9651
Feature statistics: X mean=0.0056, X std=0.9964


## Step 2: Run PySPIDER Sparse Regression

Run sparse_reg_bf on the synthetic data with correct parameters.

In [26]:
# Setup PySPIDER components with correct parameters
from PySPIDER.commons.sparse_reg_bf import sparse_reg_bf, Scaler, Initializer, Residual, ModelIterator, Threshold

print("True coefficients should be:", true_coeffs)

# Set up HOMOGENEOUS problem: [X | y] @ ξ = 0
# We want to find ξ = [c1, c2, ..., c10, -1] such that:
# c1*X[:,0] + c2*X[:,1] + ... + c10*X[:,9] - 1*y = 0
# This gives us: c1*X[:,0] + c2*X[:,1] + ... + c10*X[:,9] = y
# So the coefficients c1, c2, ... should be our true_coeffs

# Augment the library: theta = [X | y]
theta = np.column_stack([X, y])
print(f"Augmented library shape: {theta.shape} (original X + y column)")

# Set char_sizes: hierarchical for X columns, then 1.0 for y column  
char_sizes_hierarchical = np.array([1.0, 0.1, 0.01, 0.001, 0.0001, 0.00001, 0.000001, 0.0000001, 0.00000001, 0.000000001])

char_sizes_augmented = np.append(char_sizes_hierarchical, 1.0)  # Add char_size for y column

scaler = Scaler(sub_inds=None, char_sizes=char_sizes_augmented)
initializer = Initializer(method='power', start_k=4)  
residual = Residual('matrix_relative')
model_iterator = ModelIterator(max_k=11, backward_forward=True, max_passes=2)  
thres = Threshold('jump', n_terms=5)  # Force 5 terms (4 X columns + 1 y column)

print("Running sparse_reg_bf on homogeneous problem [X | y] @ ξ = 0...")

result = sparse_reg_bf(
    theta, scaler, initializer, residual, model_iterator, thres,
    verbose=True
)

print(f"\n=== SPARSE_REG_BF RESULTS ===")
print(f"Full coefficients (includes y): {result.xi}")
print(f"X coefficients: {result.xi[:-1]}")  # All but last coefficient
print(f"Y coefficient: {result.xi[-1]}")    # Last coefficient should be -1
print(f"Residual: {result.lambd:.2e}")

# The X coefficients should match true_coeffs when Y coefficient is -1
if abs(result.xi[-1] + 1) < 0.1:  # Check if y coefficient is approximately -1
    recovered_coeffs = -result.xi[:-1] / result.xi[-1]  # Normalize by y coefficient
    print(f"\nRecovered coefficients: {recovered_coeffs}")
    print(f"True coefficients:      {true_coeffs}")
    print(f"Max error: {np.max(np.abs(recovered_coeffs - true_coeffs)):.2e}")
else:
    print(f"\nY coefficient is not -1 (got {result.xi[-1]:.3f}), something's wrong")

True coefficients should be: [1.e+00 1.e-01 1.e-02 1.e-03 1.e-04 1.e-05 1.e-06 1.e-07 1.e-08 1.e-09]
Augmented library shape: (500, 11) (original X + y column)
Running sparse_reg_bf on homogeneous problem [X | y] @ ξ = 0...
Residual normalization: 21572115736.92796
Checking single-term residuals...
nrm[0]: 21.36536253193534
nrm[1]: 237.1533889462848
nrm[2]: 2251.8741235259113
nrm[3]: 21915.465927398378
nrm[4]: 227374.66412582263
nrm[5]: 2325860.9429221014
nrm[6]: 22274895.325329017
nrm[7]: 224184315.05541658
nrm[8]: 2097928444.7041776
nrm[9]: 21468677494.629787
nrm[10]: 21.58117209517257
Initializing solution with starting k: 4
mu: 0.0354462141161817 , # of iterations:  2
initial xi: [-7.071e-01 -7.068e-03 -7.059e-05  0.000e+00  0.000e+00  0.000e+00
  0.000e+00  0.000e+00  0.000e+00  0.000e+00  7.071e-01]
initial lambda: 0.01543244191265195
max_k set to 11
Iterating to find model...
Terms: [np.int64(0), np.int64(1), np.int64(2), np.int64(10)]
Setting best state for the first time...
Di

## Step 3: Apply Recursive Peeling

Now let's use the recursive peeling algorithm to improve upon the sparse regression result.

In [30]:
# Apply recursive peeling
# Create fresh components for peeling to avoid state issues
peeling_scaler = Scaler(sub_inds=None, char_sizes=char_sizes_augmented)
peeling_initializer = Initializer(method='power', start_k=4)
peeling_residual = Residual('matrix_relative')
peeling_model_iterator = ModelIterator(max_k=12, backward_forward=True, max_passes=2)
peeling_threshold = Threshold('jump', n_terms=8)  # Use fixed number instead of gamma

# Try calling without the inhomog parameter
result_2 = recursive_peeling_regression(
    theta, peeling_scaler, peeling_initializer, peeling_residual, 
    peeling_model_iterator, peeling_threshold, result,
    verbose=True
)

print(f"\n=== RECURSIVE PEELING RESULTS ===")
print(f"Original coefficients: {result.xi}")
print(f"Original residual:     {result.lambd:.2e}")
print(f"")
print(f"Peeled coefficients:   {result_2.xi}")
print(f"Peeled residual:       {result_2.lambd:.2e}")




=== STARTING PEELING REFINEMENT STEP ===
Lock strategy: Lock all non-zero terms from sparse_reg_bf
Initial residual: 7.45e-14
Initial active terms: 5
Previously locked terms: []
Newly locked terms: [0, 1, 2, 3, 10]
Total locked terms: 5
Valid locked terms (within library bounds): 5
Locking equation: 1.000*T[0] + 0.100*T[1] + 0.010*T[2] + 0.001*T[3] + -1.000*T[10]
Added RHS column for 5 locked terms
RHS column index: 11
RHS vector norm: 2.27e-03
Remaining unlocked terms: 6 -> indices: [4, 5, 6, 7, 8, 9]
RHS char_size: 2.27e-03
Scaler sub_inds set to: [4, 5, 6, 7, 8, 9, 11] (excludes locked terms [0, 1, 2, 3, 10])
Running inhomogeneous regression...
Residual normalization: 21572115736.91671
Checking single-term residuals...
nrm[0]: 227374.66412582263
nrm[1]: 2325860.942922101
nrm[2]: 22274895.32532902
nrm[3]: 224184315.05541658
nrm[4]: 2097928444.7041774
nrm[5]: 21468677494.62979
nrm[6]: 0.9999999999999997
Initializing solution with starting k: 4
mu: 0.012452232607789212 , # of iteration

In [29]:
# Apply recursive peeling
# Create fresh components for peeling to avoid state issues
peeling_scaler = Scaler(sub_inds=None, char_sizes=char_sizes_augmented)
peeling_initializer = Initializer(method='power', start_k=4)
peeling_residual = Residual('matrix_relative')
peeling_model_iterator = ModelIterator(max_k=12, backward_forward=True, max_passes=2)
peeling_threshold = Threshold('jump', n_terms=8)  # Use fixed number instead of gamma

# Try calling without the inhomog parameter
result_3 = recursive_peeling_regression(
    theta, peeling_scaler, peeling_initializer, peeling_residual, 
    peeling_model_iterator, peeling_threshold, result_2,
    verbose=True
)

print(f"\n=== RECURSIVE PEELING RESULTS ===")
print(f"Original coefficients: {result.xi}")
print(f"Original residual:     {result.lambd:.2e}")
print(f"")
print(f"Peeled coefficients:   {result_2.xi}")
print(f"Peeled residual:       {result_2.lambd:.2e}")




=== STARTING PEELING REFINEMENT STEP ===
Lock strategy: Lock all non-zero terms from sparse_reg_bf
Initial residual: 5.17e-12
Initial active terms: 10
Previously locked terms: [0, 1, 2, 3, 10]
Newly locked terms: [4, 5, 6, 9, 11]
Total locked terms: 10
Valid locked terms (within library bounds): 9
Locking equation: 1.000*T[0] + 0.100*T[1] + 0.010*T[2] + 0.001*T[3] + -1.000*T[10] + -0.000*T[4] + -0.000*T[5] + -0.000*T[6] + -0.000*T[9]
Added RHS column for 10 locked terms
RHS column index: 11
RHS vector norm: 4.52e-03
Remaining unlocked terms: 2 -> indices: [7, 8]
RHS char_size: 4.52e-03
Scaler sub_inds set to: [7, 8, 11] (excludes locked terms [0, 1, 2, 3, 10, 4, 5, 6, 9])
Running inhomogeneous regression...
Residual normalization: 2109872594.7828593
Checking single-term residuals...
nrm[0]: 224184315.05541664
nrm[1]: 2097928444.7041771
nrm[2]: 1.0
Initializing solution with starting k: 3
mu: 0.9987382747169163 , # of iterations:  1
initial xi: [ 1.574e-10  1.650e-12 -1.000e+00]
initial